# ACOS Extract-Classify-ACOS — Kaggle runner

Supports **English** (`rest16`, `laptop`) and **Amharic** (`amharic`) domains.

**Pipeline overview:**
1. Install dependencies
2. Locate and copy the repository
3. Patch known source-file bugs
4. *(Amharic only)* Upload your data and generate tokenized input files
5. Run Step 1 — aspect/opinion span extraction (BertForQuadABSA + CRF)
6. Generate candidate pairs from Step 1 predictions
7. Run Step 2 — category + sentiment classification

**Instructions:** Upload the project as a Kaggle dataset, enable Internet + GPU, set `DOMAIN` in the settings cell, then run all cells in order.

## 1. Install dependencies

In [ ]:
!pip install --upgrade pip --quiet
!pip install torchcrf boto3 --quiet
# transformers is needed for XLM-RoBERTa / AfriBERTa tokenizer + model loading
!pip install transformers==4.40.2 sentencepiece --quiet

In [ ]:
# Check GPU and Python environment
import os, sys, subprocess
print('Python:', sys.version)
try:
    print(subprocess.check_output(['nvidia-smi', '-L']).decode()[:500])
except Exception:
    print('nvidia-smi not available or no GPU')
import torch
print('torch.cuda.is_available():', torch.cuda.is_available())

## 2. Locate and copy the repository

In [ ]:
import os, shutil

# Primary path — adjust to match your Kaggle dataset slug
repo_src = '/kaggle/input/datasets/azariyasmekonen/acos-2021/ACOS'
if not os.path.exists(repo_src):
    for name in ['ACOS', 'acos', 'ACOS-main', 'Extract-Classify-ACOS']:
        p = os.path.join('/kaggle/input', name)
        if os.path.exists(p):
            repo_src = p
            break
    else:
        repo_src = None

if repo_src is None and os.path.exists('Extract-Classify-ACOS'):
    repo_src = os.path.abspath('Extract-Classify-ACOS')

if repo_src is None:
    raise RuntimeError(
        'Repository not found. Mount the Kaggle dataset at '
        '/kaggle/input/datasets/azariyasmekonen/acos-2021/ACOS or '
        'place Extract-Classify-ACOS/ in the working directory.'
    )

print('Found repository at', repo_src)
repo_dst = '/kaggle/working/Extract-Classify-ACOS'
if os.path.exists(repo_dst):
    shutil.rmtree(repo_dst)
shutil.copytree(repo_src, repo_dst)
print('Copied to', repo_dst)
os.chdir(repo_dst)
print('CWD:', os.getcwd())

## 3. Patch known source-file bugs

In [ ]:
def patch_file(path, old, new, description):
    with open(path, 'r', encoding='utf-8') as fh:
        src = fh.read()
    if old not in src:
        print(f'[SKIP] {description} — already patched')
        return
    with open(path, 'w', encoding='utf-8') as fh:
        fh.write(src.replace(old, new, 1))
    print(f'[OK]   {description}')

cwd = os.getcwd()

# Bug 1 — dataset_utils.py: hard-coded NFS import path
patch_file(
    os.path.join(cwd, 'dataset_utils.py'),
    "sys.path.insert(0, '/mnt/nfs-storage-titan/BERT/pytorch_pretrained_BERT')\n"
    "from pytorch_pretrained_bert.tokenization import BertTokenizer",
    "from bert_utils.tokenization import BertTokenizer",
    'dataset_utils.py — hard-coded NFS import'
)

# Bug 2 — run_step1.py: undefined ae_loss in gradient accumulation block
patch_file(
    os.path.join(cwd, 'run_step1.py'),
    "                if args.gradient_accumulation_steps > 1:\n"
    "                    loss = loss / args.gradient_accumulation_steps\n"
    "                    ae_loss = ae_loss / args.gradient_accumulation_steps",
    "                if args.gradient_accumulation_steps > 1:\n"
    "                    loss = loss / args.gradient_accumulation_steps",
    'run_step1.py — undefined ae_loss in gradient_accumulation block'
)
patch_file(
    os.path.join(cwd, 'run_step1.py'),
    "                    optimizer.backward(ae_loss)",
    "                    optimizer.backward(loss)",
    'run_step1.py — fp16 backward used ae_loss instead of loss'
)

# Bug 3 — run_step2.py: convert_examples_to_features2nd called with task_name
#          but the function signature expects output_mode
for old_call, desc in [
    (
        "eval_features = convert_examples_to_features2nd(\n"
        "            eval_examples, label_list, args.max_seq_length, tokenizer, task_name)",
        'run_step2.py — eval features: task_name -> output_mode'
    ),
    (
        "train_features = convert_examples_to_features2nd(\n"
        "        train_examples, label_list, args.max_seq_length, tokenizer, task_name)",
        'run_step2.py — train features: task_name -> output_mode'
    ),
    (
        "valid_features = convert_examples_to_features2nd(\n"
        "        valid_examples, label_list, args.max_seq_length, tokenizer, task_name)",
        'run_step2.py — valid features: task_name -> output_mode'
    ),
]:
    patch_file(os.path.join(cwd, 'run_step2.py'),
               old_call,
               old_call.replace(', task_name)', ', output_mode="classification")'),
               desc)

print('\nAll patches done.')

## 3a. XLM-RoBERTa / AfriBERTa compatibility patch

The codebase was written against a standalone BERT stack (`bert_utils/`).  
XLM-RoBERTa and AfriBERTa use a **SentencePiece tokenizer** and a different
model architecture that the original code cannot load natively.

This cell monkey-patches two things at runtime — no source files are modified:

1. **`bert_utils.tokenization.BertTokenizer`** — replaced with a thin shim that
   wraps `transformers.AutoTokenizer`. The shim exposes exactly the methods the
   pipeline calls (`convert_tokens_to_ids`, `convert_ids_to_tokens`,
   `tokenize`, `__call__`).  
2. **`modeling.BertPreTrainedModel.from_pretrained`** — replaced with a factory
   that loads the encoder weights via `transformers.AutoModel` and copies them
   into the custom `BertModel` so the ABSA heads continue to work unchanged.

The patch is a **no-op for plain BERT models** (`bert-base-*`,
`bert-large-*`) so the English domains are unaffected.

In [ ]:
# ── XLM-RoBERTa / AfriBERTa compatibility shim ────────────────────────────
# This cell must run AFTER cell 2 (repo copy) and AFTER cell 3 (source patches)
# so that the patched modules are already importable from the working directory.

import os, sys, types, importlib

# Ensure the working repo is on the path
_repo = '/kaggle/working/Extract-Classify-ACOS'
if _repo not in sys.path:
    sys.path.insert(0, _repo)


def _is_bert_model(model_name: str) -> bool:
    """Return True if the model name/path refers to a plain BERT variant."""
    name = os.path.basename(model_name.rstrip('/')).lower()
    return name.startswith('bert-') or name.startswith('bert_')


# ── 1. Tokenizer shim ─────────────────────────────────────────────────────

import bert_utils.tokenization as _tok_module
_OrigBertTokenizer = _tok_module.BertTokenizer


class _AutoTokenizerShim:
    """
    Minimal shim that exposes the BertTokenizer API expected by the pipeline
    while delegating to transformers.AutoTokenizer under the hood.
    This allows XLM-RoBERTa / AfriBERTa to be used without changing source.
    """

    def __init__(self, hf_tokenizer):
        self._tok = hf_tokenizer
        # Expose vocab so that any direct access still works
        self.vocab = {v: k for k, v in hf_tokenizer.get_vocab().items()}
        self.ids_to_tokens = {k: v for v, k in self.vocab.items()}

    @classmethod
    def from_pretrained(cls, model_name_or_path, do_lower_case=False, **kwargs):
        if _is_bert_model(model_name_or_path):
            # Fall through to the original BERT tokenizer
            return _OrigBertTokenizer.from_pretrained(
                model_name_or_path, do_lower_case=do_lower_case, **kwargs
            )
        from transformers import AutoTokenizer
        hf_tok = AutoTokenizer.from_pretrained(
            model_name_or_path, use_fast=True
        )
        return cls(hf_tok)

    # ── methods called by the pipeline ────────────────────────────────────

    def tokenize(self, text):
        return self._tok.tokenize(text)

    def convert_tokens_to_ids(self, tokens):
        """Accept a list of token strings and return a list of IDs."""
        return self._tok.convert_tokens_to_ids(tokens)

    def convert_ids_to_tokens(self, ids):
        """Accept a list of IDs and return a list of token strings."""
        return self._tok.convert_ids_to_tokens(ids)

    def __call__(self, text, **kwargs):
        return self._tok(text, **kwargs)


# Patch the module-level name and the class attribute so both
# `from bert_utils.tokenization import BertTokenizer` (already executed
# in run_step*.py) and future imports get the shim.
_tok_module.BertTokenizer = _AutoTokenizerShim
print('[patch] bert_utils.tokenization.BertTokenizer -> _AutoTokenizerShim')


# ── 2. Model loader shim ──────────────────────────────────────────────────

import modeling as _mod_module
_OrigFromPretrained = _mod_module.BertPreTrainedModel.from_pretrained.__func__


@classmethod
def _patched_from_pretrained(cls, pretrained_model_name_or_path, *inputs, **kwargs):
    """
    For BERT names: delegate to the original loader (unchanged behaviour).
    For XLM-R / AfriBERTa: load the encoder via transformers.AutoModel,
    transplant the weights into the custom BertModel that the ABSA heads sit on,
    then initialise the ABSA head weights from scratch.
    """
    import torch
    from transformers import AutoModel, AutoConfig
    import modeling as _m

    name = pretrained_model_name_or_path

    if _is_bert_model(name):
        return _OrigFromPretrained(cls, name, *inputs, **kwargs)

    # ── Load encoder via transformers ─────────────────────────────────────
    print(f'[patch] Loading {name} encoder via transformers.AutoModel ...')
    hf_config  = AutoConfig.from_pretrained(name)
    hf_encoder = AutoModel.from_pretrained(name)

    # Build a BertConfig that mirrors the HF config dimensions so the custom
    # BertModel has the right shape.
    bert_cfg = _m.BertConfig(
        vocab_size_or_config_json_file=hf_config.vocab_size,
        hidden_size=hf_config.hidden_size,
        num_hidden_layers=hf_config.num_hidden_layers,
        num_attention_heads=hf_config.num_attention_heads,
        intermediate_size=hf_config.intermediate_size,
        hidden_act=getattr(hf_config, 'hidden_act', 'gelu'),
        hidden_dropout_prob=hf_config.hidden_dropout_prob,
        attention_probs_dropout_prob=hf_config.attention_probs_dropout_prob,
        max_position_embeddings=getattr(hf_config, 'max_position_embeddings', 514),
        type_vocab_size=getattr(hf_config, 'type_vocab_size', 1),
        initializer_range=getattr(hf_config, 'initializer_range', 0.02),
    )

    # Instantiate the custom model (heads are randomly initialised)
    num_labels = kwargs.get('num_labels', 2)
    model = cls(bert_cfg, num_labels=num_labels)

    # ── Transplant encoder weights ─────────────────────────────────────────
    # HF RoBERTa/XLM-R state dict keys look like: roberta.encoder.layer.0.attention...
    # Custom BertModel keys look like:            bert.encoder.layer.0.attention...
    # We remap by replacing the leading 'roberta.' prefix with 'bert.'.
    hf_sd  = hf_encoder.state_dict()
    own_sd = model.state_dict()

    remapped = {}
    for hf_key, tensor in hf_sd.items():
        # Strip leading encoder-name prefix (e.g. 'roberta.', 'xlm-roberta.')
        for prefix in ('roberta.', 'bert.'):
            if hf_key.startswith(prefix):
                new_key = 'bert.' + hf_key[len(prefix):]
                break
        else:
            new_key = hf_key  # embeddings or pooler without prefix
            if not new_key.startswith('bert.'):
                new_key = 'bert.' + new_key

        if new_key in own_sd and own_sd[new_key].shape == tensor.shape:
            remapped[new_key] = tensor

    missing  = [k for k in own_sd if k not in remapped and 'bert.' in k]
    loaded   = len(remapped)
    total    = sum(1 for k in own_sd if 'bert.' in k)
    print(f'[patch] Transplanted {loaded}/{total} encoder parameter tensors.')
    if missing:
        print(f'[patch] Not transplanted (will use random init): {missing[:5]}{" ..." if len(missing)>5 else ""}')

    own_sd.update(remapped)
    model.load_state_dict(own_sd)
    return model


_mod_module.BertPreTrainedModel.from_pretrained = _patched_from_pretrained
print('[patch] modeling.BertPreTrainedModel.from_pretrained -> _patched_from_pretrained')
print('\nXLM-R / AfriBERTa compatibility patches applied.')

## 4. Settings

Set `DOMAIN` to one of:
- `'rest16'` — English restaurant reviews (SemEval-2016)
- `'laptop'` — English laptop reviews (SemEval-2016)
- `'amharic'` — Amharic reviews (your own annotated data required; sample files are included)

When `DOMAIN = 'amharic'`:
- `BERT_MODEL` defaults to `xlm-roberta-base` (XLM-RoBERTa), which has strong multilingual coverage including Ethiopic script and outperforms mBERT on low-resource languages.
- Alternatively set it to `castorini/afriberta_large` (AfriBERTa) for an Africa-focused model, or fall back to `bert-base-multilingual-cased` (mBERT) if you hit memory limits.
- `DO_LOWER_CASE` is set to `False` — Amharic uses Ethiopic/Ge'ez script, lowercasing is meaningless.
- You must run the **data preparation cell** (Section 4a) before training.

In [ ]:
# ── User-editable settings ─────────────────────────────────────────────────
DOMAIN = 'amharic'   # 'rest16' | 'laptop' | 'amharic'

EPOCHS_STEP1           = 10
EPOCHS_STEP2           = 10
TRAIN_BATCH_SIZE_STEP1 = 8
TRAIN_BATCH_SIZE_STEP2 = 8
OUTPUT_BASE            = '/kaggle/working'
MAX_SEQ_LENGTH         = 128
LEARNING_RATE_STEP1    = 2e-5
LEARNING_RATE_STEP2    = 5e-5

# ── Domain-specific automatic overrides ───────────────────────────────────
if DOMAIN == 'amharic':
    # XLM-RoBERTa gives the best multilingual performance for Ethiopic script.
    # Alternatives (uncomment to switch):
    #   'castorini/afriberta_large'      — Africa-focused RoBERTa (smaller, faster)
    #   'bert-base-multilingual-cased'   — mBERT fallback if GPU memory is tight
    BERT_MODEL    = 'xlm-roberta-base'
    DO_LOWER_CASE = False   # Ethiopic script has no case distinction
else:
    BERT_MODEL    = 'bert-base-uncased'
    DO_LOWER_CASE = True

lower_flag = '--do_lower_case' if DO_LOWER_CASE else ''
print(f'Domain      : {DOMAIN}')
print(f'BERT model  : {BERT_MODEL}')
print(f'Lower case  : {DO_LOWER_CASE}')

## 4a. Amharic data preparation *(skip if DOMAIN != 'amharic')*

The pipeline expects pre-processed TSV files under `tokenized_data/`.
For English these are already committed to the repo.
For Amharic you need to run the preparation script once after uploading your data.

### What you need to provide

Upload your Amharic quad-annotated TSV files as a Kaggle dataset and mount them (or copy them) so they are accessible. The expected files are:

```
data/Amharic-ACOS/amharic_quad_train.tsv
data/Amharic-ACOS/amharic_quad_dev.tsv
data/Amharic-ACOS/amharic_quad_test.tsv
```

### Annotation format

Each line: `review_text<TAB>asp_start,asp_end CATEGORY#SENTIMENT opi_start,opi_end [<TAB> more quads]`

- Spans are **0-indexed whitespace-tokenised word positions** (same as the English data).
- Use `-1,-1` for implicit aspect or opinion (no explicit span).
- Sentiment: `0`=negative, `1`=neutral, `2`=positive.
- Categories must match the list in `run_classifier_dataset_utils.py` under `domain_type == 'amharic'`.

Sample files with illustrative sentences are included in the repo under `data/Amharic-ACOS/`.
Replace them with your real annotated data before training.

In [ ]:
import os, shutil, subprocess

if DOMAIN == 'amharic':
    cwd = os.getcwd()  # /kaggle/working/Extract-Classify-ACOS

    # ── 1. Locate Amharic data ─────────────────────────────────────────────
    # Try the path where the repo dataset would include the data folder:
    amharic_data_candidates = [
        os.path.join(cwd, '..', 'data', 'Amharic-ACOS'),          # sibling of Extract-Classify-ACOS
        '/kaggle/input/amharic-acos',                              # standalone Kaggle dataset
        '/kaggle/input/datasets/azariyasmekonen/acos-2021/ACOS/data/Amharic-ACOS',
    ]
    amharic_data_dir = None
    for p in amharic_data_candidates:
        p = os.path.normpath(p)
        if os.path.isdir(p):
            amharic_data_dir = p
            break

    if amharic_data_dir is None:
        raise RuntimeError(
            'Amharic data directory not found. '
            'Upload your annotated TSVs (amharic_quad_{train,dev,test}.tsv) '
            'and mount the dataset so it appears at one of:\n  ' +
            '\n  '.join(amharic_data_candidates)
        )
    print('Amharic data found at:', amharic_data_dir)

    # ── 2. Copy data into working repo so the prep script can write outputs ─
    working_data_dir = os.path.join(cwd, 'amharic_data')
    if os.path.exists(working_data_dir):
        shutil.rmtree(working_data_dir)
    shutil.copytree(amharic_data_dir, working_data_dir)
    print('Copied data to:', working_data_dir)

    # ── 3. Add amharic label patch to run_classifier_dataset_utils.py ──────
    # The patch adds the 'amharic' branch to CategorySentiProcessor.get_labels().
    # This is already present in the committed version; the patch_file() call
    # below is a safety net in case an older version of the file was copied.
    utils_path = os.path.join(cwd, 'run_classifier_dataset_utils.py')
    with open(utils_path, 'r', encoding='utf-8') as fh:
        utils_src = fh.read()

    amharic_labels_block = '''
        elif domain_type == 'amharic':
            # Exact 23 categories derived from amharic_quad_{train,dev,test}.tsv
            l = [
                'AMBIENCE#DESIGN_FEATURES', 'AMBIENCE#GENERAL',
                'DELIVERY#GENERAL', 'DELIVERY#QUALITY',
                'DRINKS#PRICE', 'DRINKS#QUALITY',
                'FOOD#PRICE', 'FOOD#QUALITY', 'FOOD#STYLE_OPTIONS',
                'LOCATION#ACCESSIBILITY', 'LOCATION#GENERAL',
                'PRODUCT#DESIGN_FEATURES', 'PRODUCT#PRICE', 'PRODUCT#QUALITY',
                'RESTAURANT#GENERAL', 'RESTAURANT#PRICES',
                'SERVICE#GENERAL',
                'SHOP#GENERAL', 'SHOP#PRICES', 'SHOP#STYLE_OPTIONS',
                'STAFF#GENERAL', 'STAFF#QUALITY',
                'VALUE#GENERAL',
            ]
        elif domain_type == 'laptop':
'''
    if "domain_type == 'amharic'" not in utils_src:
        # Insert before the laptop branch
        utils_src = utils_src.replace(
            "        elif domain_type == 'laptop':",
            amharic_labels_block
        )
        with open(utils_path, 'w', encoding='utf-8') as fh:
            fh.write(utils_src)
        print('[OK]   run_classifier_dataset_utils.py — added amharic labels')
    else:
        print('[SKIP] run_classifier_dataset_utils.py — amharic labels already present')

    # ── 4. Run prepare_amharic.py to generate tokenized_data/ files ────────
    cmd = (
        f'python tokenized_data/prepare_amharic.py '
        f'--data_dir {working_data_dir} '
        f'--out_dir tokenized_data '
        f'--bert_model {BERT_MODEL}'
    )
    print('\nRunning data prep:\n', cmd)
    proc = subprocess.run(cmd, shell=True, check=False)
    if proc.returncode != 0:
        raise RuntimeError(f'prepare_amharic.py failed with exit code {proc.returncode}')

    # ── 5. Verify expected output files exist ──────────────────────────────
    expected = [
        'tokenized_data/amharic_train_quad_bert.tsv',
        'tokenized_data/amharic_dev_quad_bert.tsv',
        'tokenized_data/amharic_test_quad_bert.tsv',
        'tokenized_data/amharic_train_pair.tsv',
        'tokenized_data/amharic_dev_pair.tsv',
    ]
    missing = [f for f in expected if not os.path.exists(f)]
    if missing:
        raise RuntimeError('Missing tokenized data files:\n  ' + '\n  '.join(missing))
    print('\nAll tokenized data files present. Ready to train.')

else:
    print(f'DOMAIN={DOMAIN}: skipping Amharic data preparation.')

## 5. Step 1 — Aspect & Opinion Span Extraction

In [ ]:
import subprocess, os

STEP1_DIR_NAME = DOMAIN + '_1st'
step1_out = os.path.join(OUTPUT_BASE, 'output', 'Extract-Classify-QUAD', STEP1_DIR_NAME)

cmd = (
    f'python run_step1.py --task_name quad --do_train --do_eval '
    f'--domain_type {DOMAIN} --model_type quad {lower_flag} '
    f'--data_dir . --bert_model {BERT_MODEL} '
    f'--max_seq_length {MAX_SEQ_LENGTH} '
    f'--train_batch_size {TRAIN_BATCH_SIZE_STEP1} '
    f'--learning_rate {LEARNING_RATE_STEP1} '
    f'--num_train_epochs {EPOCHS_STEP1} '
    f'--output_dir {step1_out}'
)
print('Running Step 1:\n', cmd)
proc = subprocess.run(cmd, shell=True, check=False)
print('Step 1 exit code:', proc.returncode)

## 6. Generate Candidate Pairs from Step 1 Predictions

In [ ]:
# get_1st_pairs.py reads <base_dir>/output/Extract-Classify-QUAD/<domain>_1st/pred4pipeline.txt
# and writes tokenized_data/<domain>_test_pair_1st.tsv
cmd = f'python tokenized_data/get_1st_pairs.py . {DOMAIN}'
print('Running:', cmd)
proc = subprocess.run(cmd, shell=True, check=False)
print('get_1st_pairs exit code:', proc.returncode)

pair_file = os.path.join('tokenized_data', DOMAIN + '_test_pair_1st.tsv')
if os.path.exists(pair_file):
    with open(pair_file, encoding='utf-8') as f:
        lines = f.readlines()
    print(f'Generated {len(lines)} candidate pairs -> {pair_file}')
else:
    print(f'WARNING: {pair_file} not found — check Step 1 output')

## 7. Step 2 — Category & Sentiment Classification

In [ ]:
step2_out = os.path.join(OUTPUT_BASE, 'output', 'Extract-Classify-QUAD', DOMAIN + '_2nd')

cmd = (
    f'python run_step2.py --task_name categorysenti --do_train --do_eval '
    f'--domain_type {DOMAIN} --model_type categorysenti {lower_flag} '
    f'--data_dir . --bert_model {BERT_MODEL} '
    f'--max_seq_length {MAX_SEQ_LENGTH} '
    f'--train_batch_size {TRAIN_BATCH_SIZE_STEP2} '
    f'--learning_rate {LEARNING_RATE_STEP2} '
    f'--num_train_epochs {EPOCHS_STEP2} '
    f'--output_dir {step2_out}'
)
print('Running Step 2:\n', cmd)
proc = subprocess.run(cmd, shell=True, check=False)
print('Step 2 exit code:', proc.returncode)

## 8. Results

After both steps finish, the output files are at:
```
/kaggle/working/output/Extract-Classify-QUAD/<DOMAIN>_1st/   — Step 1 model + pred4pipeline.txt
/kaggle/working/output/Extract-Classify-QUAD/<DOMAIN>_2nd/   — Step 2 model + Test_results.txt + result.txt
```
Download them from the Kaggle output panel after the run completes.

In [ ]:
# Print the final test results from both steps
import os

for step_dir, label in [
    (os.path.join(OUTPUT_BASE, 'output', 'Extract-Classify-QUAD', DOMAIN + '_1st'), 'Step 1'),
    (os.path.join(OUTPUT_BASE, 'output', 'Extract-Classify-QUAD', DOMAIN + '_2nd'), 'Step 2'),
]:
    result_file = os.path.join(step_dir, 'Test_results.txt')
    if os.path.exists(result_file):
        print(f'\n=== {label} Test Results ===')
        with open(result_file, encoding='utf-8') as f:
            print(f.read())
    else:
        print(f'\n=== {label}: Test_results.txt not found at {result_file} ===')

## Notes

### Amharic-specific notes
- **Model**: `xlm-roberta-base` is the default for Amharic. It is a strong multilingual encoder
  that covers Ethiopic/Ge'ez script and consistently outperforms mBERT on low-resource languages.
  The XLM-R / AfriBERTa compatibility patch in Section 3a handles the tokenizer and weight-loading
  differences transparently — no source files need editing.
  Available options (set in `BERT_MODEL`):
  - `'xlm-roberta-base'` — recommended default (~278 M params, strong multilingual)
  - `'castorini/afriberta_large'` — Africa-focused RoBERTa, smaller and faster (~125 M params)
  - `'bert-base-multilingual-cased'` — mBERT fallback if GPU memory is tight (~178 M params)
- **Categories**: The 23 Amharic aspect categories are derived from the actual annotated TSV files
  and defined in `run_classifier_dataset_utils.py` under `domain_type == 'amharic'`. If you add
  new category labels to your data, add them to both the TSV files and this list — the lists must
  stay in sync or you will get a `KeyError` at training time.
- **Data size**: The sample TSVs in `data/Amharic-ACOS/` contain only illustrative sentences.
  Replace them with a real annotated corpus (minimum ~500 sentences recommended for reasonable results).
- **Tokenization**: The pipeline uses whitespace tokenization for span alignment, consistent with
  the English domains. Amharic words must be space-delimited in your TSV files.
- **Epochs**: More epochs are needed for a new language. Start with 10–20 for both steps.

### General notes
- Enable GPU in Kaggle notebook settings for practical training speed.
- Run cells in order — the XLM-R patch (Section 3a) must execute before the training cells.
- Download results from `/kaggle/working/output/Extract-Classify-QUAD/` after the run.